In [52]:
import json
import pandas as pd
import re

# Step 1: Load the JSON file
with open("export_145865_project-145865-at-2025-05-04-19-12-8471309f_without_history.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Step 2: Extract "content" and "extracted_content"
filtered_data = [
    {"content": item['data'].get("content", ""), "extracted_content": item['data'].get("extracted_content", "")}
    for item in data
]

# Step 3: Convert to DataFrame
df = pd.DataFrame(filtered_data)

# Step 4: Prepare for analysis
df['content'] = df['content'].astype(str)

def extract_text_from_content(obj):
    """Recursively extract all text from extracted_content"""
    if isinstance(obj, str):
        try:
            obj = json.loads(obj)  # Try to parse if it's a JSON string
        except:
            return obj  # Return as-is if not JSON
    
    if isinstance(obj, dict):
        return ' '.join(str(extract_text_from_content(v)) for v in obj.values())
    elif isinstance(obj, list):
        return ' '.join(str(extract_text_from_content(v)) for v in obj)
    else:
        return str(obj)

# Process extracted_content to get clean text
df['extracted_text'] = df['extracted_content'].apply(extract_text_from_content)

# Step 5: Word counts
df['content_word_count'] = df['content'].apply(lambda x: len(re.findall(r'\S+', x)))
df['extracted_word_count'] = df['extracted_text'].apply(lambda x: len(re.findall(r'\S+', x)))

# Step 6: Character lengths
df['content_len'] = df['content'].apply(len)
df['extracted_content_len'] = df['extracted_text'].apply(len)

# Calculate metrics
metrics = {
    'num_records': len(df),
    'content_word_count': df['content_word_count'].sum(),
    'extracted_word_count': df['extracted_word_count'].sum(),
    'content_avg_length': df['content_len'].mean(),
    'extracted_avg_length': df['extracted_content_len'].mean()
}

print("Analysis Results:")
print(f"Number of records: {metrics['num_records']}")
print(f"Total words in 'content': {metrics['content_word_count']}")
print(f"Total words in json 'extracted content': {metrics['extracted_word_count']}")
print(f"Average length of 'content': {metrics['content_avg_length']:.2f} characters")
print(f"Average length of json 'extracted content': {metrics['extracted_avg_length']:.2f} characters")

Analysis Results:
Number of records: 450
Total words in 'content': 408068
Total words in json 'extracted content': 139751
Average length of 'content': 4487.15 characters
Average length of json 'extracted content': 1736.52 characters
